In [39]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [54]:
# Finnish FinBERT sentiment model (TurkuNLP)
MODEL_NAME = "TurkuNLP/finnish-modernbert-large"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Set device: MPS if available (Apple Silicon), else CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at TurkuNLP/finnish-modernbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
# Create pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device
)

Device set to use mps


In [ ]:
# Sample sentences (Finnish)
texts = [
    """Inderesin kommentit ja tavoitehinnan nosto 1.05 -> 1.20€
    """,
    """Njurminenkin on valahtanut muun pölhölän mukana pohjamutiin. Mä meinasin tätä ostaaki jo tossa 0.82, mut eihän tässä enää mitään uskalla ostaa. Pitää vaan ihmetellä ja voivootella. 😊
    """,
    "Yrityksen tulos oli odotettua parempi.",
    "Nokia antoi eilen iltapäivällä positiivisen tulosvaroituksen.",
    "Euroopan keskuspankki päätti pitää ohjauskoron ennallaan.",
    "Nokia julkaisi tänään tiedotteen loppuvuoden osingonjaosta."
]


In [57]:
# Run classification
results = sentiment_pipeline(texts)

for text, res in zip(texts, results):
    print(f"{text} → {res}")

Inderesin kommentit ja tavoitehinnan nosto 1.05 -> 1.20€
     → {'label': 'LABEL_1', 'score': 0.6974858045578003}
Njurminenkin on valahtanut muun pölhölän mukana pohjamutiin. Mä meinasin tätä ostaaki jo tossa 0.82, mut eihän tässä enää mitään uskalla ostaa. Pitää vaan ihmetellä ja voivootella. 😊
     → {'label': 'LABEL_1', 'score': 0.8243575692176819}
Yrityksen tulos oli odotettua parempi. → {'label': 'LABEL_1', 'score': 0.6903408765792847}
Nokia antoi eilen iltapäivällä positiivisen tulosvaroituksen. → {'label': 'LABEL_1', 'score': 0.7888931632041931}
Euroopan keskuspankki päätti pitää ohjauskoron ennallaan. → {'label': 'LABEL_1', 'score': 0.8413879871368408}
Nokia julkaisi tänään tiedotteen loppuvuoden osingonjaosta. → {'label': 'LABEL_1', 'score': 0.7306832671165466}


In [61]:
# MPS Version
from transformers import AutoTokenizer, BertForSequenceClassification
import timeit
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
model = BertForSequenceClassification.from_pretrained("bert-base-cased").to(torch.device("mps"))

tokens = tokenizer.tokenize("Hello world, this is michael!")
tids = tokenizer.convert_tokens_to_ids(tokens)
with torch.no_grad():
    t_tids = torch.tensor([tids]*64, device=torch.device("mps"))
    res = timeit.timeit(lambda: model(input_ids=t_tids), number=100)
print(res)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


4.93062091601314


In [62]:
# MPS Version
from transformers import AutoTokenizer, BertForSequenceClassification
import timeit
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
model = BertForSequenceClassification.from_pretrained("bert-base-cased").to(torch.device("cpu"))

tokens = tokenizer.tokenize("Hello world, this is michael!")
tids = tokenizer.convert_tokens_to_ids(tokens)
with torch.no_grad():
    t_tids = torch.tensor([tids]*64, device=torch.device("cpu"))
    res = timeit.timeit(lambda: model(input_ids=t_tids), number=100)
print(res)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


12.145827000000281
